# S9 - Observabilidad con Grafana y costos

**Actividad:** instrumentar el pipeline de S8 (`atmos-eventos`) con campos de observabilidad calculados en Spark (`isValid`, `processedAt`, `latencyMs`), conectar Kafka a Prometheus, armar un tablero real en Grafana, calcular latencia agregada por ventana de tiempo, y proponer alertas y una estimacion de costos basadas en datos reales.

## Objetivo

1. Reanudar Kafka y el productor/consumidor de S7, y levantar Prometheus + Grafana unidos a la misma red.
2. Crear la `SparkSession` con el conector de Kafka (el mismo arranque de S8).
3. Leer `atmos-eventos` y calcular tres columnas nuevas: `isValid`, `processedAt`, `latencyMs`.
4. Ver los eventos en consola, con sus campos de observabilidad.
5. Revisar throughput y lag con `lastProgress`.
6. Persistir la evidencia observable en Parquet, en paralelo a la consulta de consola.

## Entorno

Este notebook asume que ya completaste 3.1-3.3 de la guia (Kafka, uso-atmos con el productor corriendo en vivo, y obs/ con Prometheus+Grafana levantado) y que estas corriendo esta celda desde JupyterLab, osea con `pyspark` ya unido a `lambda26-kafka-net` (`docker compose -f compose.yml -f compose.kafka.yml up -d`, ver `pyspark/README.md`). Si algo de eso falta, el onboarding completo vive en la guia y en el README de cada carpeta (`kafka/`, `uso-atmos/`, `obs/`), no aqui.

## 3.4 Crear el notebook y la SparkSession, con el conector de Kafka

**Producto del paso:** notebook `09_observabilidad_pipeline_grafana_costos.ipynb` con una SparkSession capaz de leer y escribir Kafka - el mismo arranque que ya hiciste en S8.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion9-observabilidad")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4040")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c60d30c3-eb39-4c0a-bf32-10dd2460dffd;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central


	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 337ms :: artifacts dl 15ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 from central in [default]
	org.slf4j#slf4j-api;2.0.17 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.8 from central in [default]
	---------------------------------------------------------------------
	|                  |       

26/10/13 14:47:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## 3.5 Leer el topic y calcular los campos de observabilidad

**Producto del paso:** un DataFrame con los campos de atmos-eventos ya tipados, mas tres columnas nuevas que no existian en S8: isValid, processedAt y latencyMs.

In [2]:
from pyspark.sql.functions import col, from_json, current_timestamp, unix_millis
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("temperatura", DoubleType())
    .add("humedad", DoubleType())
    .add("presion", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "lambda26-kafka:9092")
    .option("subscribe", "atmos-eventos")
    .option("startingOffsets", "latest")
    .load()
)

eventos = (
    crudo.select(
        col("topic"), col("partition"), col("offset"),
        col("timestamp").alias("kafkaTimestamp"),
        col("value").cast("string").alias("json_str"),
    )
    .select(
        "topic", "partition", "offset", "kafkaTimestamp",
        from_json(col("json_str"), esquema_evento).alias("e"),
    )
    .select("topic", "partition", "offset", "kafkaTimestamp", "e.*")
)

observables = (
    eventos
    .withColumn(
        "isValid",
        col("tipoEvento").isNotNull()
        & col("sensorId").isNotNull()
        & col("temperatura").isNotNull()
        & col("timestamp").isNotNull(),
    )
    .withColumn("processedAt", unix_millis(current_timestamp()))
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))
)
observables.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = false)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)



kafkaTimestamp (metadata de Kafka: cuando el broker recibio el mensaje) es distinto de timestamp (un campo del propio evento: cuando el sensor midio, igual que en S8). processedAt es el momento en que esta celda de Spark proceso el evento - y latencyMs es la diferencia entre ambos: cuanto tardo un evento en viajar de producer_sensores.py, por Kafka, hasta este punto del pipeline. isValid reusa la misma idea de contrato minimo que ya viste en S7 (consumer_sensores.py), pero calculada en Spark en vez de en Python simple.

## 3.6 Ver los eventos en consola, con sus campos de observabilidad

**Producto del paso:** la salida de consola de S8, ahora con isValid, processedAt y latencyMs visibles fila por fila.

In [3]:
# Momento 1: arrancar la consulta
consulta = (
    observables.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .start()
)

In [4]:
import time
time.sleep(25)  # deja correr la consulta de 3.6 unos micro-lotes con datos en vivo
print("micro-lotes procesados hasta ahora:", len(consulta.recentProgress))

-------------------------------------------
Batch: 0
-------------------------------------------
+-----+---------+------+--------------+----------+--------+-----------+-------+-------+------+---------+-------+-----------+---------+
|topic|partition|offset|kafkaTimestamp|tipoEvento|sensorId|temperatura|humedad|presion|origen|timestamp|isValid|processedAt|latencyMs|
+-----+---------+------+--------------+----------+--------+-----------+-------+-------+------+---------+-------+-----------+---------+
+-----+---------+------+--------------+----------+--------+-----------+-------+-------+------+---------+-------+-----------+---------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11382 |2026-10-13 14:47:24.749|sensor.lectura|esp32-laboratorio|17.2       |53.8   |1002.7 |uso-atmos|1791902844749|true   |1791902845013|264      |
|atmos-eventos|2        |23770 |2026-10-13 14:47:24.745|sensor.lectura|esp32-patio      |30.1       |32.8   |1012.5 |uso-atmos|1791902844745|true   |1791902845013|268      |
|atmos-eventos|2        |23771 |2

-------------------------------------------
Batch: 2
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11383 |2026-10-13 14:47:27.755|sensor.lectura|esp32-laboratorio|17.3       |54.0   |1002.6 |uso-atmos|1791902847755|true   |1791902850019|2264     |
|atmos-eventos|2        |23772 |2026-10-13 14:47:27.751|sensor.lectura|esp32-patio      |30.1       |33.2   |1012.6 |uso-atmos|1791902847751|true   |1791902850019|2268     |
|atmos-eventos|2        |23773 |2

-------------------------------------------
Batch: 3
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11384 |2026-10-13 14:47:30.763|sensor.lectura|esp32-laboratorio|17.3       |53.1   |1002.8 |uso-atmos|1791902850763|true   |1791902855018|4255     |
|atmos-eventos|0        |11385 |2026-10-13 14:47:33.774|sensor.lectura|esp32-laboratorio|17.3       |53.6   |1002.6 |uso-atmos|1791902853774|true   |1791902855018|1244     |
|atmos-eventos|2        |23774 |2

-------------------------------------------
Batch: 4
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11386 |2026-10-13 14:47:36.779|sensor.lectura|esp32-laboratorio|17.3       |54.0   |1002.3 |uso-atmos|1791902856779|true   |1791902860008|3229     |
|atmos-eventos|0        |11387 |2026-10-13 14:47:39.795|sensor.lectura|esp32-laboratorio|17.4       |54.2   |1002.3 |uso-atmos|1791902859794|true   |1791902860008|214      |
|atmos-eventos|2        |23778 |2

-------------------------------------------
Batch: 5
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11388 |2026-10-13 14:47:42.805|sensor.lectura|esp32-laboratorio|17.5       |54.5   |1002.8 |uso-atmos|1791902862805|true   |1791902865005|2200     |
|atmos-eventos|2        |23782 |2026-10-13 14:47:42.794|sensor.lectura|esp32-patio      |30.3       |34.2   |1012.1 |uso-atmos|1791902862793|true   |1791902865005|2212     |
|atmos-eventos|2        |23783 |2

micro-lotes procesados hasta ahora: 6


Dejala corriendo - no la detengas todavia. Vas a usarla en 3.7 para revisar lastProgress, y en 3.8 vas a arrancar una segunda consulta que persiste en Parquet al mismo tiempo. Las vas a detener juntas al final de 3.8.

## 3.7 Revisar throughput y lag con lastProgress

**Producto del paso:** una lectura real de consulta.lastProgress, con el throughput del ultimo micro-lote y el offset de cada particion de Kafka.

In [5]:
import json

progreso = consulta.lastProgress

if progreso is None:
    print("Todavia no hay micro-lote procesado. Espera unos segundos y corre esta celda de nuevo.")
else:
    fuente = progreso["sources"][0]
    metricas = fuente.get("metrics", {})
    estado = "idle" if progreso["numInputRows"] == 0 else "processed"

    log = {
        "service": "spark-streaming",
        "component": "observabilidad",
        "topic": "atmos-eventos",
        "batchId": progreso["batchId"],
        "numInputRows": progreso["numInputRows"],
        "inputRowsPerSecond": progreso["inputRowsPerSecond"],
        "processedRowsPerSecond": progreso["processedRowsPerSecond"],
        "avgOffsetsBehindLatest": metricas.get("avgOffsetsBehindLatest"),
        "maxOffsetsBehindLatest": metricas.get("maxOffsetsBehindLatest"),
        "status": estado,
    }
    print(json.dumps(log, indent=2))

{
  "service": "spark-streaming",
  "component": "observabilidad",
  "topic": "atmos-eventos",
  "batchId": 5,
  "numInputRows": 3,
  "inputRowsPerSecond": 0.6,
  "processedRowsPerSecond": 4.5,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0",
  "status": "processed"
}


numInputRows es cuantos eventos trajo el ultimo micro-lote; inputRowsPerSecond/processedRowsPerSecond son la velocidad de entrada y de procesamiento - si processedRowsPerSecond es mayor que inputRowsPerSecond, Spark esta procesando mas rapido de lo que llegan los datos, que es la situacion sana. avgOffsetsBehindLatest y maxOffsetsBehindLatest son el lag visto desde el lado de Spark: cuanto offset de Kafka le falta por leer a esta consulta en este momento. Si numInputRows sale 0, no es un error - significa que ese micro-lote en particular no encontro eventos nuevos (status: "idle").

Evidencia alternativa - Spark UI: abre http://localhost:4040 y entra a Structured Streaming -> Streaming Query Statistics. Ahi ves los mismos numeros (Input Rate, Process Rate, Batch Duration) de forma grafica, sin tener que leer lastProgress a mano.

## 3.8 Persistir la evidencia observable en Parquet

**Producto del paso:** los eventos con sus campos de observabilidad aterrizando en disco, al mismo tiempo que la consulta de consola de 3.6 sigue corriendo.

In [6]:
# Momento 1: arrancar el stream a Parquet (la consulta de 3.6 sigue corriendo en paralelo)
ARTIFACTS = "./artifacts/atmos_eventos_observabilidad"
CHECKPOINT_PARQUET = "./artifacts/chk_observabilidad"

consulta2 = (
    observables.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", ARTIFACTS)
    .option("checkpointLocation", CHECKPOINT_PARQUET)
    .trigger(processingTime="5 seconds")
    .start()
)

In [7]:
import os

def tamano_bytes(ruta):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(ruta) for f in fs if f.endswith(".parquet"))

time.sleep(60)
t1, filas1, bytes1 = time.time(), spark.read.parquet(ARTIFACTS).count(), tamano_bytes(ARTIFACTS)
print(f"medición 1: {filas1} filas, {bytes1/1024:.1f} KB")
time.sleep(120)
t2, filas2, bytes2 = time.time(), spark.read.parquet(ARTIFACTS).count(), tamano_bytes(ARTIFACTS)
print(f"medición 2: {filas2} filas, {bytes2/1024:.1f} KB  ({t2 - t1:.0f} s después)")

-------------------------------------------
Batch: 6
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11389 |2026-10-13 14:47:45.811|sensor.lectura|esp32-laboratorio|17.5       |54.4   |1003.1 |uso-atmos|1791902865811|true   |1791902870005|4194     |
|atmos-eventos|0        |11390 |2026-10-13 14:47:48.817|sensor.lectura|esp32-laboratorio|17.8       |54.1   |1003.3 |uso-atmos|1791902868817|true   |1791902870005|1188     |
|atmos-eventos|2        |23784 |2

-------------------------------------------
Batch: 7
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11391 |2026-10-13 14:47:51.826|sensor.lectura|esp32-laboratorio|17.7       |53.3   |1003.3 |uso-atmos|1791902871826|true   |1791902875005|3179     |
|atmos-eventos|0        |11392 |2026-10-13 14:47:54.844|sensor.lectura|esp32-laboratorio|17.7       |52.8   |1003.3 |uso-atmos|1791902874844|true   |1791902875005|161      |
|atmos-eventos|2        |23788 |2

-------------------------------------------
Batch: 8
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11393 |2026-10-13 14:47:57.851|sensor.lectura|esp32-laboratorio|17.8       |53.4   |1003.1 |uso-atmos|1791902877851|true   |1791902880019|2168     |
|atmos-eventos|2        |23792 |2026-10-13 14:47:57.847|sensor.lectura|esp32-patio      |30.1       |36.0   |1012.6 |uso-atmos|1791902877847|true   |1791902880019|2172     |
|atmos-eventos|2        |23793 |2

-------------------------------------------
Batch: 9
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11394 |2026-10-13 14:48:00.86 |sensor.lectura|esp32-laboratorio|17.9       |52.5   |1002.8 |uso-atmos|1791902880860|true   |1791902885009|4149     |
|atmos-eventos|0        |11395 |2026-10-13 14:48:03.874|sensor.lectura|esp32-laboratorio|17.7       |52.3   |1002.9 |uso-atmos|1791902883874|true   |1791902885009|1135     |
|atmos-eventos|2        |23794 |2

-------------------------------------------
Batch: 10
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11396 |2026-10-13 14:48:06.89 |sensor.lectura|esp32-laboratorio|17.8       |52.8   |1002.8 |uso-atmos|1791902886890|true   |1791902890008|3118     |
|atmos-eventos|0        |11397 |2026-10-13 14:48:09.897|sensor.lectura|esp32-laboratorio|17.9       |52.7   |1003.2 |uso-atmos|1791902889896|true   |1791902890008|112      |
|atmos-eventos|2        |23798 |

...

-------------------------------------------
Batch: 39
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11444 |2026-10-13 14:50:31.241|sensor.lectura|esp32-laboratorio|18.3       |52.7   |1000.8 |uso-atmos|1791903031241|true   |1791903035019|3778     |
|atmos-eventos|0        |11445 |2026-10-13 14:50:34.25 |sensor.lectura|esp32-laboratorio|18.3       |53.2   |1000.5 |uso-atmos|1791903034250|true   |1791903035019|769      |
|atmos-eventos|2        |23894 |

-------------------------------------------
Batch: 40
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11446 |2026-10-13 14:50:37.255|sensor.lectura|esp32-laboratorio|18.4       |52.9   |1000.4 |uso-atmos|1791903037255|true   |1791903040003|2748     |
|atmos-eventos|2        |23898 |2026-10-13 14:50:37.252|sensor.lectura|esp32-patio      |28.1       |38.1   |1012.6 |uso-atmos|1791903037252|true   |1791903040003|2751     |
|atmos-eventos|2        |23899 |

-------------------------------------------
Batch: 41
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11447 |2026-10-13 14:50:40.267|sensor.lectura|esp32-laboratorio|18.6       |52.8   |1000.6 |uso-atmos|1791903040267|true   |1791903045006|4739     |
|atmos-eventos|0        |11448 |2026-10-13 14:50:43.269|sensor.lectura|esp32-laboratorio|18.5       |52.7   |1000.3 |uso-atmos|1791903043269|true   |1791903045006|1737     |
|atmos-eventos|2        |23900 |

-------------------------------------------
Batch: 42
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11449 |2026-10-13 14:50:46.274|sensor.lectura|esp32-laboratorio|18.6       |53.5   |1000.1 |uso-atmos|1791903046274|true   |1791903050004|3730     |
|atmos-eventos|0        |11450 |2026-10-13 14:50:49.282|sensor.lectura|esp32-laboratorio|18.7       |53.1   |1000.6 |uso-atmos|1791903049282|true   |1791903050004|722      |
|atmos-eventos|2        |23904 |

medición 2: 174 filas, 302.4 KB  (122 s después)


Nota el nombre consulta2, no consulta: la consulta de 3.6 sigue viva en la variable consulta, y si esta nueva reusara ese mismo nombre, perderias la unica referencia que te permite detenerla mas tarde.

Mientras corre, puedes leer lo que ya escribio sin detenerla - este paso es opcional:

In [8]:
# Momento 2 (opcional): leer lo guardado (se puede repetir, el stream sigue corriendo)
guardado = spark.read.parquet(ARTIFACTS)
total = guardado.count()
print("filas guardadas:", total)
guardado.select("sensorId", "latencyMs", "isValid").show(10, truncate=False)

-------------------------------------------
Batch: 43
-------------------------------------------
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento    |sensorId         |temperatura|humedad|presion|origen   |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+--------------+-----------------+-----------+-------+-------+---------+-------------+-------+-------------+---------+
|atmos-eventos|0        |11451 |2026-10-13 14:50:52.289|sensor.lectura|esp32-laboratorio|18.6       |52.8   |1000.8 |uso-atmos|1791903052289|true   |1791903055010|2721     |
|atmos-eventos|2        |23908 |2026-10-13 14:50:52.286|sensor.lectura|esp32-patio      |27.6       |38.1   |1012.5 |uso-atmos|1791903052285|true   |1791903055010|2725     |
|atmos-eventos|2        |23909 |

filas guardadas: 180


+-----------------+---------+-------+
|sensorId         |latencyMs|isValid|
+-----------------+---------+-------+
|esp32-patio      |3186     |true   |
|esp32-invernadero|3181     |true   |
|esp32-patio      |176      |true   |
|esp32-invernadero|164      |true   |
|esp32-patio      |4819     |true   |
|esp32-invernadero|4818     |true   |
|esp32-patio      |1818     |true   |
|esp32-invernadero|1814     |true   |
|esp32-patio      |3995     |true   |
|esp32-invernadero|3993     |true   |
+-----------------+---------+-------+
only showing top 10 rows


Cuando ya viste suficiente, deten las dos consultas juntas - la de consola de 3.6 y la de Parquet de este paso:

In [9]:
# Momento 3: detener ambas consultas (la de consola de 3.6 y la de Parquet de 3.8)
historial = consulta.recentProgress + consulta2.recentProgress
consulta.stop()
consulta2.stop()
print("consultas detenidas; micro-lotes registrados:", len(historial))

consultas detenidas; micro-lotes registrados: 81


26/10/13 14:50:56 ERROR FileFormatWriter: Aborting job 2bf2e152-2701-43b8-b5ef-a3d9dc4f5290.
java.lang.InterruptedException
	at java.base/java.util.concurrent.locks.AbstractQueuedSynchronizer.tryAcquireSharedNanos(AbstractQueuedSynchronizer.java:1167)
	at scala.concurrent.impl.Promise$DefaultPromise.tryAwait0(Promise.scala:251)
	at scala.concurrent.impl.Promise$DefaultPromise.result(Promise.scala:271)
	at org.apache.spark.util.SparkThreadUtils$.awaitResultNoSparkExceptionConversion(SparkThreadUtils.scala:78)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.B

Error frecuente: si el log muestra lineas ERROR WriteToDataSourceV2Exec, TaskKilledException o Aborting task al llamar .stop(), es el mismo ruido benigno que ya viste en S8 - Spark cancelando una tarea a mitad de un micro-lote, no una falla real.

## 3.9 Analizar la latencia agregada por ventana de tiempo

**Producto del paso:** una tabla con la latencia promedio, mínima y máxima por minuto, calculada sobre la evidencia real guardada en 3.8.

In [10]:
from pyspark.sql.functions import window, avg, min as spark_min, max as spark_max, count, round as sround

guardado = spark.read.parquet(ARTIFACTS)

latencia_por_minuto = (
    guardado.groupBy(window(col("kafkaTimestamp"), "1 minute"))
    .agg(
        count("*").alias("numEventos"),
        sround(avg("latencyMs"), 1).alias("avgLatencyMs"),
        spark_min("latencyMs").alias("minLatencyMs"),
        spark_max("latencyMs").alias("maxLatencyMs"),
    )
    .select(
        col("window.start").alias("inicioVentana"),
        col("window.end").alias("finVentana"),
        "numEventos", "avgLatencyMs", "minLatencyMs", "maxLatencyMs",
    )
    .orderBy("inicioVentana")
)
latencia_por_minuto.show(20, truncate=False)

resumen = guardado.agg(
    count("*").alias("eventos"),
    sround(avg("latencyMs"), 1).alias("avgLatencyMs"),
    spark_max("latencyMs").alias("maxLatencyMs"),
    sround(avg(col("isValid").cast("int")) * 100, 2).alias("pctValidos"),
).collect()[0]
print("Resumen de la evidencia:", resumen.asDict())

+-------------------+-------------------+----------+------------+------------+------------+
|inicioVentana      |finVentana         |numEventos|avgLatencyMs|minLatencyMs|maxLatencyMs|
+-------------------+-------------------+----------+------------+------------+------------+
|2026-10-13 14:47:00|2026-10-13 14:48:00|9         |1840.3      |161         |3186        |
|2026-10-13 14:48:00|2026-10-13 14:49:00|60        |2065.9      |2           |4162        |
|2026-10-13 14:49:00|2026-10-13 14:50:00|60        |2922.4      |868         |4970        |
|2026-10-13 14:50:00|2026-10-13 14:51:00|51        |2726.7      |722         |4819        |
+-------------------+-------------------+----------+------------+------------+------------+



Resumen de la evidencia: {'eventos': 180, 'avgLatencyMs': 2527.3, 'maxLatencyMs': 4970, 'pctValidos': 100.0}


La latencia varía dentro de cada minuto aunque el sistema esté sano: el productor publica cada 3 s (`SENSOR_INTERVAL_MS`) y Spark dispara cada 5 s, así que la espera de un evento hasta el próximo disparo cicla entre ~0 y ~5 s cada 15 s (mínimo común múltiplo).

## 3.10 Alertas propuestas y umbrales

**Producto del paso:** las alertas de la Tabla 8, con el valor real medido hoy junto a cada umbral. Los valores de infraestructura se consultan a Prometheus desde el propio notebook (`http://prometheus:9090`, misma red Docker).

In [11]:
import urllib.request, urllib.parse

def promql(expr):
    url = "http://prometheus:9090/api/v1/query?" + urllib.parse.urlencode({"query": expr})
    datos = json.load(urllib.request.urlopen(url, timeout=10))["data"]["result"]
    return [(r["metric"], float(r["value"][1])) for r in datos]

brokers = promql("kafka_brokers")[0][1]
exporter_up = promql('up{job="kafka-exporter"}')[0][1]
lag_max_10m = max(v for _, v in promql('max_over_time(sum by (consumergroup) (kafka_consumergroup_lag{consumergroup="uso-atmos-group"})[10m:15s])'))
lat = latencia_por_minuto.agg(spark_max("avgLatencyMs").alias("m")).collect()[0]["m"]

alertas = [
    ("Kafka no visible", "kafka_brokers < 1", f"kafka_brokers = {brokers:.0f}", "Prometheus/Grafana", "OK" if brokers >= 1 else "ALERTA"),
    ("Exporter caído", 'up{job="kafka-exporter"} == 0', f"up = {exporter_up:.0f}", "Prometheus/Grafana", "OK" if exporter_up == 1 else "ALERTA"),
    ("Lag alto sostenido", "kafka_consumergroup_lag > 100 (2 min)", f"máx. lag 10 min = {lag_max_10m:.0f}", "Prometheus/Grafana", "OK" if lag_max_10m <= 100 else "ALERTA"),
    ("Latencia alta sensible", "avgLatencyMs > 3000 por minuto", f"peor minuto = {lat:.0f} ms", "Spark/Parquet", "OK" if lat <= 3000 else "ALERTA"),
    ("Latencia alta crítica", "avgLatencyMs > 6000 por minuto", f"peor minuto = {lat:.0f} ms", "Spark/Parquet", "OK" if lat <= 6000 else "ALERTA"),
]
print(f"{'Situación':<24}{'Regla propuesta':<42}{'Valor medido hoy':<26}{'Dónde':<20}Estado")
for a in alertas:
    print(f"{a[0]:<24}{a[1]:<42}{a[2]:<26}{a[3]:<20}{a[4]}")

Situación               Regla propuesta                           Valor medido hoy          Dónde               Estado
Kafka no visible        kafka_brokers < 1                         kafka_brokers = 1         Prometheus/Grafana  OK
Exporter caído          up{job="kafka-exporter"} == 0             up = 1                    Prometheus/Grafana  OK
Lag alto sostenido      kafka_consumergroup_lag > 100 (2 min)     máx. lag 10 min = 6       Prometheus/Grafana  OK
Latencia alta sensible  avgLatencyMs > 3000 por minuto            peor minuto = 2922 ms     Spark/Parquet       OK
Latencia alta crítica   avgLatencyMs > 6000 por minuto            peor minuto = 2922 ms     Spark/Parquet       OK


**Por qué estos umbrales y no los de ejemplo de la guía.** La guía sugiere `avgLatencyMs > 100 / > 1000`, pero con un *trigger* de 5 s la espera estructural de un evento ya es de 0 a 5 s: un umbral de 100 ms o 1000 ms estaría disparando siempre con el sistema sano. Se fijaron justo por encima de lo que el diseño explica: **> 3000 ms** (más de la mitad del *trigger* en promedio de un minuto entero) como aviso, y **> 6000 ms** (más que un *trigger* completo: los eventos ya esperan más de un micro-lote) como crítico. El *lag* medido nunca pasó de unos pocos mensajes, así que **100** queda dos órdenes de magnitud por encima del ruido normal.

## 3.11 Estimación de costos y plan de escalado

**Producto del paso:** la Tabla 9 completada con los valores medidos en esta corrida, y el costo mensual estimado con los precios de referencia de la guía (AWS us-east-1: `t3.medium` ≈ US$ 0.0416/h, S3 Standard ≈ US$ 0.023/GB-mes).

In [12]:
pico_eps = max((p["inputRowsPerSecond"] for p in historial if p["numInputRows"] > 0), default=0)
prom_eps = sum(p["numInputRows"] for p in historial) / max(1, len(historial)) / 5
crec_bytes_h = (bytes2 - bytes1) / (t2 - t1) * 3600
crec_filas_h = (filas2 - filas1) / (t2 - t1) * 3600

PRECIO_T3_MEDIUM_H = 0.0416
PRECIO_S3_GB_MES = 0.023
HORAS_MES = 730
instancias = 3  # broker Kafka (+ exporter), script de streaming, obs/ (Prometheus + Grafana)
costo_computo = instancias * PRECIO_T3_MEDIUM_H * HORAS_MES
gb_mes = crec_bytes_h * HORAS_MES / 1024**3
costo_almacenamiento = gb_mes * PRECIO_S3_GB_MES

print("Tabla 9 - valores medidos en esta sesión")
print(f"  Eventos por segundo (pico, inputRowsPerSecond) : {pico_eps:.2f}")
print(f"  Eventos por segundo (promedio)                  : {prom_eps:.2f}")
print(f"  Latencia promedio por minuto (peor minuto)      : {lat:.0f} ms")
print(f"  Crecimiento del Parquet                         : {crec_filas_h:,.0f} filas/h  |  {crec_bytes_h/1024**2:.2f} MB/h  ->  {gb_mes:.2f} GB/mes")
print()
print("Costo mensual estimado (24/7)")
print(f"  Cómputo: {instancias} x t3.medium x {HORAS_MES} h x US$ {PRECIO_T3_MEDIUM_H} = US$ {costo_computo:.2f}")
print(f"  Almacenamiento: {gb_mes:.2f} GB/mes x US$ {PRECIO_S3_GB_MES} = US$ {costo_almacenamiento:.4f} (primer mes; se acumula sin política de retención)")
print(f"  TOTAL ≈ US$ {costo_computo + costo_almacenamiento:.2f}/mes")

Tabla 9 - valores medidos en esta sesión
  Eventos por segundo (pico, inputRowsPerSecond) : 1.80
  Eventos por segundo (promedio)                  : 0.96
  Latencia promedio por minuto (peor minuto)      : 2922 ms
  Crecimiento del Parquet                         : 3,545 filas/h  |  6.03 MB/h  ->  4.30 GB/mes

Costo mensual estimado (24/7)
  Cómputo: 3 x t3.medium x 730 h x US$ 0.0416 = US$ 91.10
  Almacenamiento: 4.30 GB/mes x US$ 0.023 = US$ 0.0989 (primer mes; se acumula sin política de retención)
  TOTAL ≈ US$ 91.20/mes


**Plan de escalado (umbrales de decisión).**

1. **Lag sostenido > 100 mensajes durante 2 min** o `processedRowsPerSecond` < `inputRowsPerSecond` en varios micro-lotes seguidos: el consumidor no da abasto. Primera acción: subir el paralelismo de Spark (más núcleos, `spark.sql.shuffle.partitions`) y aumentar particiones del topic — hoy la partición 1 de `atmos-eventos` no recibe datos, así que repartir mejor las *keys* es lo más barato.
2. **avgLatencyMs > 6000 ms por minuto** con lag bajo: el cuello está en el micro-lote (duración del lote ≥ *trigger*). Acción: acortar el *trigger* o separar la escritura a Parquet en otra consulta/instancia.
3. **Si el volumen se duplica**, lo primero que se escala es el **consumidor (Spark)**: con ~1 evento/s el broker y el Parquet tienen holgura de varios órdenes de magnitud, y el costo de almacenamiento sigue siendo de centavos; el cómputo de 3 instancias domina el costo total.